# 07b IQ Blobs (Raw ADC) – Execute

Runs the experiment on the QOP and saves `ds_raw` to a local `.nc` file. Use notebook 02 to load and analyze.

## Imports

In [ ]:
import os
import numpy as np
import xarray as xr
from qm.qua import *

from qualang_tools.multi_user import qm_session
from qualang_tools.results import progress_counter
from qualang_tools.units import unit

from iqcc_calibration_tools.qualibrate_config.qualibrate.node import QualibrationNode
from quam_builder.architecture.superconducting.qpu import FluxTunableQuam as Quam
from calibration_utils.iq_blobs_raw_adc import Parameters
from qualibration_libs.parameters import get_qubits
from qualibration_libs.data import XarrayDataFetcher

## Node initialisation

In [ ]:
description = """
IQ BLOBS (RAW ADC)
Uses raw ADC traces instead of on-board measure and demodulation.
Measures the resonator N times in |g> state and |e> state.
"""

node = QualibrationNode[Parameters, Quam](
    name="07b_iq_blobs_raw_adc",
    description=description,
    parameters=Parameters(),
)

@node.run_action(skip_if=node.modes.external)
def custom_param(node: QualibrationNode[Parameters, Quam]):
    node.parameters.qubits = ["Q1"]
    pass

node.machine = Quam.load()

# Local path for saving ds_raw (same folder as this notebook)
DS_RAW_PATH = os.path.join(os.getcwd(), "07b_iq_blobs_raw_adc_ds_raw.nc")

## Create QUA program

In [ ]:
@node.run_action(skip_if=node.parameters.load_data_id is not None)
def create_qua_program(node: QualibrationNode[Parameters, Quam]):
    u = unit(coerce_to_integer=True)
    node.namespace["qubits"] = qubits = get_qubits(node)
    num_qubits = len(qubits)

    n_runs = node.parameters.num_shots
    operation = node.parameters.operation
    readout_length = qubits[0].resonator.operations[operation].length
    node.namespace["sweep_axes"] = {
        "qubit": xr.DataArray(qubits.get_names()),
        "n_runs": xr.DataArray(np.linspace(1, n_runs, n_runs), attrs={"long_name": "number of shots"}),
        "readout_time": xr.DataArray(
            np.arange(0, readout_length, 1),
            attrs={"long_name": "readout time", "units": "ns"},
        ),
    }

    with program() as node.namespace["qua_program"]:
        n = declare(int)
        n_st = declare_stream()
        adc_g_st = [declare_stream(adc_trace=True) for _ in range(num_qubits)]
        adc_e_st = [declare_stream(adc_trace=True) for _ in range(num_qubits)]

        for multiplexed_qubits in qubits.batch():
            for qubit in multiplexed_qubits.values():
                node.machine.initialize_qpu(target=qubit)
            align()

            with for_(n, 0, n < n_runs, n + 1):
                save(n, n_st)
                for i, qubit in multiplexed_qubits.items():
                    qubit.reset(node.parameters.reset_type, node.parameters.simulate)
                align()
                for i, qubit in multiplexed_qubits.items():
                    reset_if_phase(qubit.resonator.name)
                    qubit.resonator.measure(operation, stream=adc_g_st[i])
                    qubit.resonator.wait(qubit.resonator.depletion_time * u.ns)
                align()

                for i, qubit in multiplexed_qubits.items():
                    qubit.reset(node.parameters.reset_type, node.parameters.simulate)
                align()
                for i, qubit in multiplexed_qubits.items():
                    qubit.xy.play("x180")
                    qubit.align()
                    reset_if_phase(qubit.resonator.name)
                    qubit.resonator.measure(operation, stream=adc_e_st[i])
                    qubit.resonator.wait(qubit.resonator.depletion_time * u.ns)
                align()

        with stream_processing():
            n_st.save("n")
            for i in range(num_qubits):
                qubit = node.namespace["qubits"][i]
                if qubit.resonator.opx_input.port_id == 1:
                    stream_g = adc_g_st[i].input1()
                    stream_e = adc_e_st[i].input1()
                else:
                    stream_g = adc_g_st[i].input2()
                    stream_e = adc_e_st[i].input2()
                stream_g.real().buffer(n_runs).save(f"Igraw{i + 1}")
                stream_g.image().buffer(n_runs).save(f"Qgraw{i + 1}")
                stream_e.real().buffer(n_runs).save(f"Ieraw{i + 1}")
                stream_e.image().buffer(n_runs).save(f"Qeraw{i + 1}")

create_qua_program(node)

## Execute & save locally

Runs the experiment on the QOP and saves `ds_raw` to the node and to a local `.nc` file.

In [ ]:
if not node.parameters.simulate and node.parameters.load_data_id is None:
    qmm = node.machine.connect()
    config = node.machine.generate_config()
    with qm_session(qmm, config, timeout=node.parameters.timeout) as qm:
        node.namespace["job"] = job = qm.execute(node.namespace["qua_program"])
        data_fetcher = XarrayDataFetcher(job, node.namespace["sweep_axes"])
        for dataset in data_fetcher:
            progress_counter(
                data_fetcher["n"],
                node.parameters.num_shots,
                start_time=data_fetcher.t_start,
            )
        node.log(job.execution_report())

    node.results["ds_raw"] = dataset
    dataset.to_netcdf(DS_RAW_PATH)
    print(f"Saved ds_raw to {DS_RAW_PATH}")
else:
    print("Skipped execution (simulate=True or load_data_id set)")